# 08. Quality Filter Validation

This notebook validates the new spectral quality filter that checks for biological signatures:
1. **Lipid peak (2800-3000 cm⁻¹)** is the HIGHEST peak
2. **Most signal < 0.15** (baseline/noise level)
3. **Clear biological peaks** (not flat noise)
4. **Reasonable SNR** (not all zeros or saturated)

We'll demonstrate which spectra pass/fail the filter.

In [ ]:
import sys
import os
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Setup
project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
if project_root not in sys.path:
    sys.path.append(project_root)

from src.data.preprocessing import RamanPreprocessor
import pandas as pd

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (14, 5)
print(" Imports complete")

In [ ]:
# Load sample raw data and preprocess
freq_path = "../data/RamanFreqTable.csv"
df_freq = pd.read_csv(freq_path)
wavenumbers = df_freq.iloc[:, 1].values

print(f"Wavenumber range: {wavenumbers[0]:.1f} - {wavenumbers[-1]:.1f} cm⁻¹")
print(f"Number of calibration points: {len(wavenumbers)}")

# Initialize preprocessor to use its quality check method
preprocessor = RamanPreprocessor(
    raw_data_dir="../data/raw",
    output_file="../data/test_output.h5",
    calibration_path=freq_path,
    use_ramanspy=False
)

print(f"\n Preprocessor initialized")
print(f"  Bio-mask bands: {preprocessor.idx_bio_start}-{preprocessor.idx_bio_end}")
print(f"  HWN-mask bands: {preprocessor.idx_hwn_start}-{preprocessor.idx_hwn_end}")

In [ ]:
# Load sample spectra from Classes 3 & 4
import glob

class_3_files = sorted(glob.glob("../data/raw/LGm-3/*.npy"))[:3]
class_4_files = sorted(glob.glob("../data/raw/LGm-4/*.npy"))[:3]

print(f"Loading {len(class_3_files)} Class 3 files and {len(class_4_files)} Class 4 files...\n")

# Extract some spectra and preprocess them
test_spectra = []
test_quality = []
test_labels = []

from scipy.ndimage import gaussian_filter

for class_id, files in [(3, class_3_files), (4, class_4_files)]:
    for file_path in files:
        try:
            raw_cube = np.load(file_path)
            if isinstance(raw_cube, np.ndarray) and raw_cube.dtype == 'O':
                raw_cube = raw_cube.item().get('data', raw_cube)
            
            if raw_cube.ndim != 3:
                continue
            
            # Get a few random pixels
            smoothed_cube = gaussian_filter(raw_cube, sigma=(1.0, 1.0, 0))
            
            # Extract bio-mask to get valid tissue region
            valid_mask = preprocessor._create_bio_mask(smoothed_cube)
            if np.sum(valid_mask) == 0:
                continue
            
            y_coords, x_coords = np.where(valid_mask)
            flat_data = raw_cube[valid_mask]
            
            # Preprocess and check quality
            for spectrum in flat_data[:20]:  # Sample first 20 from each file
                normalized_spec = preprocessor._preprocess_spectrum(spectrum)
                passes_quality = preprocessor._is_spectrum_quality_good(normalized_spec)
                
                test_spectra.append(normalized_spec)
                test_quality.append(passes_quality)
                test_labels.append(f"Class {class_id}")
        
        except Exception as e:
            print(f"Error processing {os.path.basename(file_path)}: {e}")

test_spectra = np.array(test_spectra)
test_quality = np.array(test_quality)
test_labels = np.array(test_labels)

n_pass = np.sum(test_quality)
n_fail = len(test_quality) - n_pass

print(f"\nProcessed {len(test_spectra)} spectra")
print(f"  Passed quality check: {n_pass} ({100*n_pass/len(test_quality):.1f}%)")
print(f"  Failed quality check: {n_fail} ({100*n_fail/len(test_quality):.1f}%)")

In [ ]:
# Visualize: Good vs Bad spectra
good_indices = np.where(test_quality)[0]
bad_indices = np.where(~test_quality)[0]

fig, axes = plt.subplots(2, 2, figsize=(14, 8))

# Good spectra examples
ax = axes[0, 0]
if len(good_indices) > 0:
    for idx in good_indices[:5]:
        ax.plot(test_spectra[idx], alpha=0.5, linewidth=1)
    ax.set_title(f' Good Spectra (n={len(good_indices)})', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No good spectra found', ha='center', va='center')
    ax.set_title('No good spectra', fontsize=12)
ax.set_xlabel('Wavenumber Index')
ax.set_ylabel('Intensity (normalized)')
ax.set_ylim([0, 1])
ax.grid(alpha=0.3)

# Bad spectra examples
ax = axes[0, 1]
if len(bad_indices) > 0:
    for idx in bad_indices[:5]:
        ax.plot(test_spectra[idx], alpha=0.5, linewidth=1, color='red')
    ax.set_title(f' Bad Spectra (n={len(bad_indices)})', fontsize=12, fontweight='bold')
else:
    ax.text(0.5, 0.5, 'No bad spectra found', ha='center', va='center')
    ax.set_title('No bad spectra', fontsize=12)
ax.set_xlabel('Wavenumber Index')
ax.set_ylabel('Intensity (normalized)')
ax.set_ylim([0, 1])
ax.grid(alpha=0.3)

# Quality metrics comparison
ax = axes[1, 0]
if len(good_indices) > 0 and len(bad_indices) > 0:
    good_max = np.max(test_spectra[good_indices], axis=1)
    bad_max = np.max(test_spectra[bad_indices], axis=1)
    
    ax.hist(good_max, bins=20, alpha=0.6, label='Good', color='green', edgecolor='black')
    ax.hist(bad_max, bins=20, alpha=0.6, label='Bad', color='red', edgecolor='black')
    ax.set_xlabel('Max Intensity')
    ax.set_ylabel('Count')
    ax.set_title('Distribution of Max Intensity')
    ax.legend()
    ax.grid(alpha=0.3)

# Summary text
ax = axes[1, 1]
ax.axis('off')
summary_text = f"""
QUALITY FILTER RESULTS:

Total spectra: {len(test_spectra)}
Passed filter: {n_pass} ({100*n_pass/len(test_quality):.1f}%)
Rejected: {n_fail} ({100*n_fail/len(test_quality):.1f}%)

Quality checks:
• Lipid peak is highest
• Max intensity 0.05-0.98
• >60% of signal < 0.15
• Clear peak structure
• Good SNR (std > 0.02)
"""
ax.text(0.1, 0.5, summary_text, fontsize=11, family='monospace',
        verticalalignment='center', bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))

plt.tight_layout()
plt.savefig("../results/data_quality/08_quality_filter_validation.png", dpi=150, bbox_inches='tight')
plt.show()

print(f"\n Saved quality filter validation plot")

In [ ]:
# Detailed analysis: What causes rejection?
print("\n" + "="*80)
print("DETAILED QUALITY FILTER ANALYSIS")
print("="*80)

if len(bad_indices) > 0:
    print(f"\nAnalyzing {min(5, len(bad_indices))} rejected spectra:\n")
    
    for i, idx in enumerate(bad_indices[:5]):
        spec = test_spectra[idx]
        
        # Check each criterion
        max_intensity = np.max(spec)
        mean_intensity = np.mean(spec)
        std_intensity = np.std(spec)
        pct_low = 100 * np.sum(spec < 0.15) / len(spec)
        q25 = np.percentile(spec, 25)
        q75 = np.percentile(spec, 75)
        
        # Check lipid peak
        lipid_region = spec[preprocessor.idx_hwn_start:preprocessor.idx_hwn_end]
        interior_region = spec[len(spec)//20:-len(spec)//20]
        lipid_max = np.max(lipid_region) if len(lipid_region) > 0 else 0
        interior_max = np.max(interior_region) if len(interior_region) > 0 else 0
        
        print(f"  Spectrum {i+1} from {test_labels[idx]}:")
        print(f"    Max intensity: {max_intensity:.4f} {' too low' if max_intensity < 0.05 else ' too high' if max_intensity > 0.98 else ''}")
        print(f"    Mean: {mean_intensity:.4f}, Std: {std_intensity:.4f}")
        print(f"    % low signal (<0.15): {pct_low:.1f}% {' too high' if pct_low < 60 else ''}")
        print(f"    Signal range (Q25-Q75): {q75-q25:.4f} {' too small' if (q75-q25) < 0.03 else ''}")
        print(f"    Lipid max: {lipid_max:.4f}, Interior max: {interior_max:.4f}", end="")
        if lipid_max < interior_max * 0.8:
            print(f"  lipid not dominant")
        elif lipid_max < 0.3:
            print(f"  lipid peak too weak")
        else:
            print(f" ")
        print()